In [ ]:
# only run this code for cluster 
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset
import torch.nn.functional as torch_F # avoid import problems becasue F is used as a variable

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *
from utils import *

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)
biopsy_df = dfs['biopsy']

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)

notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')

all_valid_patient_ids = get_valid_patient_ids(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    min_ts_count=10,
    require_notes=True,
)
with open('../data/splits/pool_assignments.json') as f:
    pool_assignments = json.load(f)
pool_a_ids = np.asarray(pool_assignments['pool_a'])
pool_a_set = set(pool_a_ids.tolist())
selected_patient_ids = np.asarray([pid for pid in all_valid_patient_ids if pid in pool_a_set])

global_split_path = '../data/splits/global_split_pool_a_9010.json'
full_split_ids = get_or_create_global_split(
    patient_ids=selected_patient_ids,
    split_json_path=global_split_path,
    train_size=0.9,
    val_size=0.1,
    test_size=0.0,
    random_state=42,
    shuffle=True,
    force_recreate=False,
)

train_dataset = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=full_split_ids['train'],
    fit_preprocessing=True,
    min_ts_count=10,
    require_notes=True,
)
preprocessing_artifacts = train_dataset.preprocessing_artifacts

val_dataset = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=full_split_ids['val'],
    preprocessing_artifacts=preprocessing_artifacts,
    fit_preprocessing=False,
    min_ts_count=10,
    require_notes=True,
)

ts_scaler = train_dataset.ts_scaler
static_scaler = train_dataset.scaler

train_ids = set(full_split_ids['train'])
val_ids = set(full_split_ids['val'])
selected_ids = set(selected_patient_ids.tolist())

assert train_ids.isdisjoint(val_ids), 'Data leakage: train overlaps val'
assert (train_ids | val_ids) == selected_ids, 'Split IDs do not cover selected cohort exactly'

print(f"Eligible Pool A size: {len(selected_patient_ids)}")
print(f"Global split sizes (90/10): {len(train_ids)} / {len(val_ids)}")
print(f"Using shared split file: {global_split_path}")
print('Validation split is used only for model checkpointing.')
print(f"Categorical cardinalities from Pool A train split: {train_dataset.categorical_cardinalities}")

In [ ]:
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# vanilla_lstm = VanillaTimeSeriesEncoder()
att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(att_encoder, categorical_cardinalities=train_dataset.categorical_cardinalities, use_static=True, use_notes=True).to(device)
predict_steps_ahead = 1
lambda_corr = 0.0
lambda_modality_mi = 0.0
lambda_feature_loss = 0.0


In [ ]:
criterion = nn.MSELoss(reduction='none')
optimizer = optim.Adam(model.parameters(), lr=0.0003)

feature_decoders = FeatureDecoders(CONFIG['lstm_hidden_size'], CONFIG).to(device)
feature_decoder_optimizer = optim.Adam(feature_decoders.parameters(), lr=0.0003)

num_epochs = 30
min_delta = 1e-4
best_val_loss = float('inf')
best_epoch = -1
save_path = '../models/backbone_poola_9010_best.pt'

for epoch in range(num_epochs):
    epoch_loss = 0.0
    total_valid_points = 0.0
    model.train()

    with tqdm(total=len(train_dataloader), desc=f'Train Epoch {epoch+1}/{num_epochs}', leave=False) as pbar:
        for batch in train_dataloader:
            cat_features = batch['static_categorical_features'].to(device)
            num_features_input = batch['static_numerical_features'].to(device)

            full_seq = batch['ts_features'].to(device)
            timesteps = batch['timesteps'].to(device)
            value_mask_full = batch['value_mask'].to(device)
            mask = batch['mask'].to(device)
            B, T, F = full_seq.shape

            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps = batch['notes_timesteps'].to(device)
            notes_mask = batch['notes_mask'].to(device)

            if T <= predict_steps_ahead:
                pbar.update(1)
                continue

            input_seq = full_seq[:, :-predict_steps_ahead, :]
            input_timesteps = timesteps[:, :-predict_steps_ahead]
            input_mask = mask[:, :-predict_steps_ahead]
            elapsed_times = build_elapsed_times(input_timesteps, input_mask)
            input_value_mask = value_mask_full[:, :-predict_steps_ahead, :]

            optimizer.zero_grad()
            feature_decoder_optimizer.zero_grad()

            outputs, hidden_states, _, static_embedding = model(
                x=input_seq,
                elapsed_times=elapsed_times,
                timesteps=input_timesteps,
                notes_embeddings=notes_embeddings,
                notes_timesteps=notes_timesteps,
                static_features=(cat_features, num_features_input),
                mask=input_mask,
                value_mask=input_value_mask,
                notes_mask=notes_mask
            )

            target_seq = full_seq[:, predict_steps_ahead:, :]
            valid_mask = value_mask_full[:, predict_steps_ahead:, :]

            raw_loss = criterion(outputs, target_seq)
            masked_loss = raw_loss * valid_mask.float()
            valid_points = valid_mask.float().sum()

            accum_loss = masked_loss.sum()
            final_loss = accum_loss / valid_points if valid_points > 0 else torch.tensor(0.0, device=device)

            last_hidden = get_last_valid_step(hidden_states, input_mask)
            last_notes = get_last_valid_note_embedding(notes_embeddings, notes_mask)
            last_ts = get_last_valid_step(full_seq, mask)
            decorr_loss = correlation_loss(last_hidden)

            mi_loss = modality_mi_loss(
                last_hidden,
                static_embedding,
                last_notes
            )

            if lambda_feature_loss > 0.0:
                feature_preds, masks = feature_decoders(last_hidden)
                feat_pred_loss = feature_prediction_loss(
                    feature_preds,
                    masks,
                    static_target=static_embedding,
                    notes_target=last_notes,
                    ts_target=last_ts
                )
            else:
                feat_pred_loss = torch.tensor(0.0, device=device)

            total_loss = final_loss + lambda_corr * decorr_loss + lambda_modality_mi * mi_loss + lambda_feature_loss * feat_pred_loss
            total_loss.backward()
            optimizer.step()
            feature_decoder_optimizer.step()

            epoch_loss += masked_loss.sum().item()
            total_valid_points += valid_points.item()

            pbar.update(1)
            pbar.set_postfix({
                'Loss': f'{total_loss.item():.4f}',
                'MSE': f'{final_loss.item():.4f}'
            })

    avg_train_loss = epoch_loss / total_valid_points if total_valid_points > 0 else 0.0
    print(f"Epoch {epoch+1}/{num_epochs} Train Loss: {avg_train_loss:.4f}")

    model.eval()
    val_loss_accum = 0.0
    val_valid_points = 0.0

    with torch.no_grad():
        with tqdm(total=len(val_dataloader), desc=f'Val Epoch {epoch+1}/{num_epochs}', leave=False) as pbar_val:
            for batch in val_dataloader:
                cat_features = batch['static_categorical_features'].to(device)
                num_features_input = batch['static_numerical_features'].to(device)

                full_seq = batch['ts_features'].to(device)
                timesteps = batch['timesteps'].to(device)
                value_mask_full = batch['value_mask'].to(device)
                mask = batch['mask'].to(device)
                B, T, F = full_seq.shape

                notes_embeddings = batch['notes_embeddings'].to(device)
                notes_timesteps = batch['notes_timesteps'].to(device)
                notes_mask = batch['notes_mask'].to(device)

                if T <= predict_steps_ahead:
                    pbar_val.update(1)
                    continue

                input_seq = full_seq[:, :-predict_steps_ahead, :]
                input_timesteps = timesteps[:, :-predict_steps_ahead]
                input_mask = mask[:, :-predict_steps_ahead]
                elapsed_times = build_elapsed_times(input_timesteps, input_mask)
                input_value_mask = value_mask_full[:, :-predict_steps_ahead, :]

                outputs, _, _, _ = model(
                    x=input_seq,
                    elapsed_times=elapsed_times,
                    timesteps=input_timesteps,
                    notes_embeddings=notes_embeddings,
                    notes_timesteps=notes_timesteps,
                    static_features=(cat_features, num_features_input),
                    mask=input_mask,
                    value_mask=input_value_mask,
                    notes_mask=notes_mask
                )

                target_seq = full_seq[:, predict_steps_ahead:, :]
                valid_mask = value_mask_full[:, predict_steps_ahead:, :]

                raw_loss = criterion(outputs, target_seq)
                masked_loss = raw_loss * valid_mask.float()
                valid_points = valid_mask.float().sum()

                val_loss_accum += masked_loss.sum().item()
                val_valid_points += valid_points.item()

                pbar_val.update(1)

    avg_val_loss = val_loss_accum / val_valid_points if val_valid_points > 0 else float('inf')
    print(f"Epoch {epoch+1}/{num_epochs} Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss + min_delta < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model at epoch {best_epoch} -> {save_path}")

print(f"Training completed. Best epoch: {best_epoch}, Best val loss: {best_val_loss:.4f}")

if os.path.exists(save_path):
    model.load_state_dict(torch.load(save_path, map_location=device))
    print(f"Training completed. Loaded best checkpoint from epoch {best_epoch}: {save_path}")
else:
    print('Warning: No best checkpoint saved; using last epoch weights.')

In [ ]:
#! Visualizing - optional for making TFN
model.eval()
with torch.no_grad():
    for batch in train_dataloader:
        cat_features = batch['static_categorical_features'].to(device)
        num_features = batch['static_numerical_features'].to(device)

        full_seq = batch['ts_features'].to(device)       # (B, T, F)
        timesteps = batch['timesteps'].to(device)        # (B, T)
        value_mask_full = batch['value_mask'].to(device) # (B, T, F)

        notes_embeddings = batch['notes_embeddings'].to(device)
        notes_timesteps = batch['notes_timesteps'].to(device)
        notes_mask = batch['notes_mask'].to(device)

        B, T, F = full_seq.shape

        # Skip if sequence is too short for the desired horizon
        if T <= predict_steps_ahead:
            continue

        # Input => (B, T - n, F)
        input_seq = full_seq[:, :-predict_steps_ahead, :]
        # Target => (B, T - n, F)
        target_seq = full_seq[:, predict_steps_ahead:, :]

        input_mask = batch['mask'][:, :-predict_steps_ahead].to(device)
        elapsed_times = build_elapsed_times(timesteps[:, :-predict_steps_ahead], input_mask)

        input_value_mask = value_mask_full[:, :-predict_steps_ahead, :]

        # Value mask => for the same target portion
        value_mask = value_mask_full[:, predict_steps_ahead:, :]

        outputs, _, _, _ = model(
            x=input_seq,
            elapsed_times=elapsed_times,
            timesteps=timesteps[:, :-predict_steps_ahead],
            notes_embeddings=notes_embeddings,
            notes_timesteps=notes_timesteps,
            static_features=(cat_features, num_features),
            mask=input_mask,
            value_mask=input_value_mask,
            notes_mask=notes_mask
        )
        # outputs => (B, T - n, F), normalized

        outputs_np = outputs.cpu().numpy()
        target_np = target_seq.cpu().numpy()
        value_mask_np = value_mask.cpu().numpy()
        timesteps_np = timesteps.cpu().numpy()

        B_, TmN, F_ = outputs_np.shape  # TmN = T - n

        outputs_2d = outputs_np.reshape(-1, F_)
        target_2d = target_np.reshape(-1, F_)

        # Apply inverse scaling
        outputs_orig = ts_scaler.inverse_transform(outputs_2d).reshape(B_, TmN, F_)
        target_orig = ts_scaler.inverse_transform(target_2d).reshape(B_, TmN, F_)

        patient_idx = 2
        pred_values = outputs_orig[patient_idx]    # shape (T - n, F)
        true_values = target_orig[patient_idx]     # shape (T - n, F)
        mask_vals = value_mask_np[patient_idx]     # shape (T - n, F)

        # Time steps for those predicted points: (T - n) steps from index n onward
        patient_timesteps = timesteps_np[patient_idx, predict_steps_ahead:]  # shape (T - n,)

        feature_idx = list(range(len(CONFIG['ts_features'])))
        num_features_to_plot = len(feature_idx)
        fig, axs = plt.subplots(num_features_to_plot, 1, figsize=(10, 6 * num_features_to_plot))

        # Handle single axis vs multiple
        if num_features_to_plot == 1:
            axs = [axs]

        for i, f_idx in enumerate(feature_idx):
            valid_indices = (mask_vals[:, f_idx] == 1)

            axs[i].scatter(patient_timesteps[valid_indices],
                           true_values[valid_indices, f_idx],
                           color='blue', label='Actual', alpha=0.7)
            axs[i].plot(patient_timesteps[valid_indices],
                        true_values[valid_indices, f_idx],
                        color='blue', alpha=0.3, linestyle='--')

            axs[i].scatter(patient_timesteps[valid_indices],
                           pred_values[valid_indices, f_idx],
                           color='red', label='Predicted', alpha=0.7)
            axs[i].plot(patient_timesteps[valid_indices],
                        pred_values[valid_indices, f_idx],
                        color='red', alpha=0.3, linestyle='--')

            feature_name = CONFIG['ts_features'][f_idx]
            axs[i].set_title(f"{feature_name} Predictions vs Actual (N-step={predict_steps_ahead})")
            axs[i].set_xlabel("Relative Time (days)")
            axs[i].set_ylabel(feature_name)
            axs[i].legend()

        plt.tight_layout()
        plt.show()

        # Just visualize for the first batch and stop
        break